# Complexhibit — Modelo base y comparación con el sistema final

Entrena los tres niveles del sistema con el modelo **XLM-RoBERTa base**, de menor tamaño. Fue la primera versión completa del sistema y se conserva para compararla con el **sistema final**, entrenado con XLM-RoBERTa large (cuaderno `01_entrenamiento_etiquetador`).

| Nivel | Qué extrae | Método |
|---|---|---|
| 1. Bloques | `catalogueItem` (obra del catálogo) y `exhibition` | Modelo *transformer* (BIO) |
| 2. Entidades | persona, título, institución/sede, lugar, fecha, técnica/material, dimensiones, dirección, nº de inventario | Modelo *transformer* (BIO) |
| 3. Detalle | alto/ancho/fondo/peso y unidad; fechas ISO (`when`, `from`/`to`) | Reglas |
| 3. Detalle | rol de persona, tipo y rol de institución, tipo de obra, tipo de técnica | Clasificador *transformer* con contexto |

Los tres niveles se combinan en una salida anidada (obra → título, técnica, fecha, dimensiones desglosadas…), según el manual de etiquetado.

**Entrada:** un fichero ZIP con los ficheros XML anotados en GATE (carpeta `XML_GATE`).

**Requisitos:** entorno con GPU (T4). Duración aproximada: 1 a 1,5 horas.

**Salida:** un ZIP con métricas, informes, partición y salidas JSON por nivel.

## 0. Configuración

In [ ]:
# Entrada: ZIP con los XML de GATE; los resultados se guardan en el entorno de ejecución.
from google.colab import files
import os, glob, zipfile, collections

print('Entrada: fichero ZIP con los XML de GATE (carpeta XML_GATE)')
subidos = files.upload()
os.makedirs('/content/xml', exist_ok=True)
for nombre in subidos:
    if nombre.lower().endswith('.zip'):
        zipfile.ZipFile(nombre).extractall('/content/xml')
    elif nombre.lower().endswith('.xml'):
        os.replace(nombre, '/content/xml/' + nombre)

CARPETA_XML    = '/content/xml'
CARPETA_SALIDA = '/content/resultados_ner'
CARPETA_TXT    = '/content/txt'          # opcional: no se usa en esta versión salvo que subas los TXT aquí
EPOCAS_N2, EPOCAS_N1, EPOCAS_ATR = 8, 5, 8

def es_gate(p):
    try:
        with open(p, encoding='utf-8', errors='ignore') as f:
            return 'GateDocument' in f.read(400)
    except Exception:
        return False
xmls = [p for p in glob.glob(CARPETA_XML + '/**/*.xml', recursive=True) if es_gate(p)]
assert xmls, 'No hay ficheros de GATE en el ZIP subido'
MODELOS = CARPETA_SALIDA + '/modelos'
os.makedirs(MODELOS, exist_ok=True)
print(len(xmls), 'ficheros de GATE listos')

In [ ]:
!pip -q install -U transformers datasets seqeval accelerate

Módulo con la lectura de GATE, la conversión a BIO, las reglas del nivel 3 y el montaje de la salida anidada.

In [ ]:
%%writefile complexhibit_ie.py
"""Complexhibit: extracción de información de catálogos en tres niveles.

Nivel 1  bloques:   catalogueItem (ITEM) y exhibition (EXHIB)        -> modelo BIO
Nivel 2  entidades: persName, title, institution, placeName, date...  -> modelo BIO
Nivel 3  detalle:   dimensiones y fechas                              -> reglas
                    roles y tipos (person.role, institution.type...)  -> clasificador

Este módulo contiene todo lo que no depende del modelo: lectura de GATE,
conversión a BIO, reglas, ejemplos para el clasificador de atributos,
decodificación de predicciones y montaje de la salida anidada.
"""
import json
import re
import xml.etree.ElementTree as ET
from fractions import Fraction
from pathlib import Path

# ----------------------------------------------------------------------------
# Esquema (según el Manual de etiquetado v1)
# ----------------------------------------------------------------------------
# El orden de cada diccionario fija la prioridad si dos anotaciones se solapan.
NIVEL2 = {
    "persName": "PER",
    "title": "TITLE",
    "institution": "ORG",
    "venue": "ORG",
    "placeName": "LOC",
    "date": "DATE",
    "technique": "TECH",
    "material": "TECH",
    "dimensions": "DIM",
    "address": "ADDR",
    "idno": "IDNO",
}
NIVEL1 = {
    "catalogueItem": "ITEM",
    "exhibition": "EXHIB",
}
# Atributos que aprende el clasificador del nivel 3: (tipo GATE, rasgo)
ATRIBUTOS = [
    ("person", "role"),
    ("institution", "type"),
    ("institution", "role"),
    ("work", "type"),
    ("technique", "type"),
]

# Reemplazo global erróneo detectado en el corpus: al renombrar valores de
# atributos (city -> poblation, material -> technique) se modificó también
# el texto. Se corrige solo el texto de cada palabra; las posiciones no cambian.
TOKEN_FIXES = [
    (re.compile(r"poblation"), "city"),
    (re.compile(r"techniqueity"), "materiality"),
    (re.compile(r"techniquely"), "materially"),
    (re.compile(r"detechniqueiz"), "dematerializ"),
]

TOKEN_RE = re.compile(r"\w+(?:[-'’]\w+)*|[^\w\s]", re.UNICODE)


# ----------------------------------------------------------------------------
# Lectura de GATE
# ----------------------------------------------------------------------------
def load_gate(path):
    """Devuelve (texto, anotaciones) con anotaciones = [(tipo, ini, fin, rasgos)]."""
    root = ET.parse(path).getroot()
    tw = root.find("TextWithNodes")
    text = tw.text or ""
    offsets = {}
    for node in tw:
        offsets[int(node.get("id"))] = len(text)
        text += node.tail or ""
    anns = []
    for aset in root.findall("AnnotationSet"):
        for a in aset.findall("Annotation"):
            s = offsets.get(int(a.get("StartNode")), int(a.get("StartNode")))
            e = offsets.get(int(a.get("EndNode")), int(a.get("EndNode")))
            feats = {f.find("Name").text: (f.find("Value").text or "") for f in a.findall("Feature")}
            if e > s:
                anns.append((a.get("Type"), s, e, feats))
    return text, anns


def fix_token(tok):
    for pat, rep in TOKEN_FIXES:
        tok = pat.sub(rep, tok)
    return tok


def tokenize(text):
    return [(m.group(), m.start(), m.end()) for m in TOKEN_RE.finditer(text)]


# ----------------------------------------------------------------------------
# Conversión a BIO
# ----------------------------------------------------------------------------
def resolve_overlaps(spans, prioridad):
    spans = sorted(spans, key=lambda x: (prioridad.index(x[2]), -(x[1] - x[0])))
    kept = []
    for s, e, t in spans:
        if all(e <= ks or s >= ke for ks, ke, _ in kept):
            kept.append((s, e, t))
    return sorted(kept)


def to_bio(tokens, spans, mapa):
    tags = ["O"] * len(tokens)
    for s, e, t in spans:
        inside = [i for i, (_, ts, te) in enumerate(tokens) if ts < e and te > s]
        for k, i in enumerate(inside):
            tags[i] = ("B-" if k == 0 else "I-") + mapa[t]
    return tags


def segment_bounds(tokens, text, tags=None, max_tokens=200):
    """Índices [ini, fin) de fragmentos: se corta en líneas en blanco o al llegar
    a max_tokens, sin partir nunca una entidad."""
    bounds, start = [], 0
    for i, (_, s, e) in enumerate(tokens):
        nxt = tokens[i + 1] if i + 1 < len(tokens) else None
        if nxt is None:
            bounds.append((start, i + 1))
            break
        inside_next = tags is not None and tags[i + 1].startswith("I-")
        gap = text[e:nxt[1]]
        if not inside_next and ("\n\n" in gap or i + 1 - start >= max_tokens):
            bounds.append((start, i + 1))
            start = i + 1
    return bounds


def documento_bio(path, mapa, max_tokens=200):
    text, anns = load_gate(path)
    spans = [(s, e, t) for t, s, e, _ in anns if t in mapa]
    tokens = tokenize(text)
    tags = to_bio(tokens, resolve_overlaps(spans, list(mapa)), mapa)
    filas = []
    for k, (a, b) in enumerate(segment_bounds(tokens, text, tags, max_tokens)):
        filas.append({
            "doc": Path(path).stem, "seg": k,
            "tokens": [fix_token(t[0]) for t in tokens[a:b]],
            "tags": tags[a:b],
        })
    return filas


def construir_bio(carpeta, mapa, salida, max_tokens=200):
    n, stats = 0, {}
    with open(salida, "w", encoding="utf-8") as fh:
        for p in sorted(Path(carpeta).rglob("*.xml")):
            try:
                filas = documento_bio(p, mapa, max_tokens)
            except ET.ParseError as err:
                print(f"AVISO: no se pudo leer {p.name}: {err}")
                continue
            for f in filas:
                fh.write(json.dumps(f, ensure_ascii=False) + "\n")
                for t in f["tags"]:
                    if t.startswith("B-"):
                        stats[t[2:]] = stats.get(t[2:], 0) + 1
            n += 1
    print(f"{n} documentos -> {salida}")
    print("Entidades por etiqueta:", dict(sorted(stats.items(), key=lambda x: -x[1])))
    return stats


# ----------------------------------------------------------------------------
# Nivel 3a: reglas de dimensiones
# ----------------------------------------------------------------------------
_NUM = r"\d+(?:[.,]\d+)?(?:\s+\d+/\d+)?|\d+/\d+"


def _valor(s):
    s = s.strip().replace(",", ".")
    try:
        return float(sum(Fraction(p) for p in s.split()))
    except (ValueError, ZeroDivisionError):
        return None


def parse_dimensions(texto, unidad_defecto=None):
    """'93 x 19 x 16' -> {'height': 93.0, 'width': 19.0, 'depth': 16.0, 'unit': ...}
    Convención de los catálogos: alto x ancho x fondo."""
    t = texto.lower()
    out = {}
    if re.search(r"\b(lbs?|pounds?)\b", t):
        out["unit"] = "libra"
    elif re.search(r"(?<![a-záéíóú])kg\b|kilo", t):
        out["unit"] = "kg"
    elif re.search(r"\bfeet\b|(?<![a-záéíóú])ft\b|\bpies\b", t):
        out["unit"] = "feet"
    elif re.search(r"(?<![a-záéíóú])cm\b", t):
        out["unit"] = "cm"
    elif re.search(r"(?<![a-záéíóú])mm\b", t):
        out["unit"] = "mm"
    elif re.search(r"(?<![a-záéíóú])m\b|metros", t):
        out["unit"] = "m"
    elif '"' in t or "inch" in t or "pulgada" in t or "in." in t:
        out["unit"] = "inches"
    if "unit" not in out and unidad_defecto:
        out["unit"] = unidad_defecto
    nums = re.findall(_NUM, t)
    if out.get("unit") in ("libra", "kg"):
        if nums:
            out["weight"] = _valor(nums[0])
        return out
    if re.search(r"\bdi[aá]m|ø|⌀", t) and nums:
        out["diameter"] = _valor(nums[0])
        return out
    rango = re.search(rf"({_NUM})\s+(?:to|a|-)\s+({_NUM})\s+(?:feet|ft|pies|cm|m)?\s*(?:in height|de alto|alto)", t)
    if rango:
        out["height"] = (_valor(rango.group(1)), _valor(rango.group(2)))
        return out
    if re.search(r"in height|de alto|\balto\b", t) and nums:
        out["height"] = _valor(nums[0])
        return out
    partes = re.split(r"\s*[x×]\s*", t)
    vals = [_valor(m.group()) for p in partes for m in [re.search(_NUM, p)] if m]
    for clave, v in zip(("height", "width", "depth"), vals):
        out[clave] = v
    return out


def unidad_documento(text):
    """Unidad declarada para todo el catálogo, p. ej. 'All dimensions are given
    in inches' o 'Medidas en centímetros'."""
    t = text.lower()
    m = re.search(r"\b(dimensions?|measurements?|medidas|dimensiones)\b[^.\n;]{0,30}?\b(?:in|en)\s+(inches|inch|cent[ií]met\w*|cm|metros|feet)\b", t)
    if not m:
        return None
    u = m.group(2)
    return "inches" if u.startswith("inch") else "feet" if u == "feet" else "m" if u == "metros" else "cm"


def unidad_mayoritaria(frases):
    """Unidad explícita más frecuente entre las dimensiones de un catálogo: se usa
    para las medidas que no indican unidad (p. ej. '19 5/8 x 29 13/16')."""
    cuenta = {}
    for f in frases:
        u = parse_dimensions(f).get("unit")
        if u and u not in ("libra", "kg"):   # solo unidades de longitud
            cuenta[u] = cuenta.get(u, 0) + 1
    return max(cuenta, key=cuenta.get) if cuenta else None


def unidad_por_defecto(text, frases):
    """Primero la unidad declarada en el catálogo ('dimensions are given in inches');
    si no la hay, la más frecuente entre sus medidas."""
    return unidad_documento(text) or unidad_mayoritaria(frases)


def dimensiones_oro(text, anns):
    """Pares (texto de la dimensión, valores anotados) para evaluar las reglas."""
    res = []
    for t, s, e, f in anns:
        if t != "dimensions":
            continue
        oro = {"unit": f.get("unit")} if f.get("unit") else {}
        for t2, s2, e2, _ in anns:
            if t2 in ("height", "width", "depth", "weight", "diameter") and s <= s2 and e2 <= e:
                v = text[s2:e2]
                m = re.match(rf"\s*({_NUM})\s+(?:to|a|-)\s+({_NUM})\s*$", v)
                oro[t2] = (_valor(m.group(1)), _valor(m.group(2))) if m else _valor(v)
        res.append((text[s:e], oro))
    return res


# ----------------------------------------------------------------------------
# Nivel 3b: reglas de fechas (formato ISO del manual: when / from-to)
# ----------------------------------------------------------------------------
MESES = {
    "jan": 1, "january": 1, "enero": 1, "ene": 1,
    "feb": 2, "february": 2, "febrero": 2,
    "mar": 3, "march": 3, "marzo": 3,
    "apr": 4, "april": 4, "abril": 4, "abr": 4,
    "may": 5, "mayo": 5,
    "jun": 6, "june": 6, "junio": 6,
    "jul": 7, "july": 7, "julio": 7,
    "aug": 8, "august": 8, "agosto": 8, "ago": 8,
    "sep": 9, "sept": 9, "september": 9, "septiembre": 9, "setiembre": 9,
    "oct": 10, "october": 10, "octubre": 10,
    "nov": 11, "november": 11, "noviembre": 11,
    "dec": 12, "december": 12, "diciembre": 12, "dic": 12,
}
_DIAS_MES = [31, 29, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]


def _ultimo_dia(y, m):
    if m == 2:
        return 29 if (y % 4 == 0 and (y % 100 != 0 or y % 400 == 0)) else 28
    return _DIAS_MES[m - 1]


_RANGO_INCIERTO = re.compile(r"^\W*(\d{4})\s*[-–—]\s*(\d{2}|\d{4})\s*/\s*(\d{4})\W*$")
_ANIO_ABREV = re.compile(r"\b(\d{2})(\d{2})\s*[-–—]\s*(\d{2})(?![\d/])")


def parse_date(texto):
    t = texto.lower().replace("’", "'")
    # '1920-21/1987': fecha de creación incierta (1920 o 1921) y fecha posterior (p. ej., estampación)
    m = _RANGO_INCIERTO.match(t)
    if m:
        a, b, c = m.groups()
        b = a[:2] + b if len(b) == 2 else b
        return {"notBefore": a, "notAfter": b, "to": c}
    # '1969-71' -> '1969 - 1971'
    t = _ANIO_ABREV.sub(lambda m: f"{m.group(1)}{m.group(2)} - {m.group(1)}{m.group(3)}", t)
    toks = re.findall(r"[a-záéíóúñ]+|\d+", t)
    fechas = []  # lista de [dia, mes, año] en orden de aparición
    cur = [None, None, None]
    for tok in toks:
        if tok.isdigit():
            n = int(tok)
            if len(tok) == 4 and 1000 <= n <= 2100:
                cur[2] = n
                fechas.append(cur)
                cur = [None, None, None]
            elif n <= 31:
                if cur[0] is not None:  # nuevo día sin año: cerrar el anterior
                    fechas.append(cur)
                    cur = [None, None, None]
                cur[0] = n
        elif tok in MESES:
            if cur[1] is not None:
                fechas.append(cur)
                cur = [None, None, None]
            cur[1] = MESES[tok]
    if cur != [None, None, None]:
        fechas.append(cur)
    # formatos numéricos dd/mm/aaaa
    num = [x for x in re.findall(r"(\d{1,2})[/.-](\d{1,2})[/.-](\d{4})", t) if 1 <= int(x[1]) <= 12]
    if num:
        fechas = [[int(d), int(m), int(y)] for d, m, y in num]
    fechas = [f for f in fechas if any(x is not None for x in f)]
    if not fechas:
        return {}
    # el año (y el mes) se propagan hacia atrás: "9 January-6 March, 1998"
    for i in range(len(fechas) - 2, -1, -1):
        if fechas[i][2] is None:
            fechas[i][2] = fechas[i + 1][2]
            if fechas[i][1] is None:
                fechas[i][1] = fechas[i + 1][1]
    if any(f[2] is None for f in fechas):
        return {}
    # el mes se propaga hacia delante en rangos de días: 'June 20-26, 1963'
    for i in range(1, len(fechas)):
        if fechas[i][1] is None and fechas[i][0] is not None and fechas[i - 1][1] is not None:
            fechas[i][1] = fechas[i - 1][1]

    def iso(f, fin=False):
        d, m, y = f
        if m is None:
            return f"{y:04d}" if not fin else f"{y:04d}"
        if d is None:
            d = _ultimo_dia(y, m) if fin else 1
        return f"{y:04d}-{m:02d}-{d:02d}"

    if len(fechas) == 1:
        f = fechas[0]
        if f[1] is None:
            return {"when": f"{f[2]:04d}"}
        if f[0] is None:  # mes y año: intervalo del mes
            return {"from": iso(f), "to": iso(f, fin=True)}
        return {"when": iso(f)}
    return {"from": iso(fechas[0]), "to": iso(fechas[-1], fin=True)}


def fechas_oro(text, anns):
    res = []
    for t, s, e, f in anns:
        if t == "date":
            oro = {k: v for k, v in f.items() if k in ("when", "from", "to", "notBefore", "notAfter")}
            if oro:
                res.append((text[s:e], oro))
    return res


def evaluar_reglas(carpeta, docs=None):
    """Exactitud de las reglas frente a los valores anotados a mano."""
    dim_ok = dim_n = fec_ok = fec_n = 0
    errores = {"dimensiones": [], "fechas": []}
    for p in sorted(Path(carpeta).rglob("*.xml")):
        if docs is not None and p.stem not in docs:
            continue
        try:
            text, anns = load_gate(p)
        except ET.ParseError:
            continue
        pares = dimensiones_oro(text, anns)
        udoc = unidad_por_defecto(text, [f for f, _ in pares])
        for frase, oro in pares:
            pred = parse_dimensions(frase, udoc)
            campos = [k for k in oro if k != "unit"]
            ok = all(pred.get(k) == oro[k] for k in campos) and (not oro.get("unit") or pred.get("unit") == oro["unit"])
            dim_n += 1
            dim_ok += ok
            if not ok and len(errores["dimensiones"]) < 30:
                errores["dimensiones"].append({"texto": frase, "oro": oro, "regla": pred, "doc": p.stem})
        for frase, oro in fechas_oro(text, anns):
            pred = parse_date(frase)
            ok = pred == oro
            fec_n += 1
            fec_ok += ok
            if not ok and len(errores["fechas"]) < 30:
                errores["fechas"].append({"texto": frase, "oro": oro, "regla": pred, "doc": p.stem})
    return {
        "dimensiones": {"n": dim_n, "exactitud": dim_ok / dim_n if dim_n else None},
        "fechas": {"n": fec_n, "exactitud": fec_ok / fec_n if fec_n else None},
        "errores": errores,
    }


# ----------------------------------------------------------------------------
# Nivel 3c: ejemplos para el clasificador de roles y tipos
# ----------------------------------------------------------------------------
VENTANA = 250  # caracteres de contexto a cada lado


def con_marcas(text, s, e, ventana=VENTANA):
    a, b = max(0, s - ventana), min(len(text), e + ventana)
    frag = text[a:s] + " [E] " + text[s:e] + " [/E] " + text[e:b]
    frag = " ".join(fix_token(w) for w in frag.split())
    return frag


def ejemplos_atributos(carpeta):
    filas = []
    for p in sorted(Path(carpeta).rglob("*.xml")):
        try:
            text, anns = load_gate(p)
        except ET.ParseError:
            continue
        for t, s, e, f in anns:
            for tipo, rasgo in ATRIBUTOS:
                if t == tipo and f.get(rasgo, "").strip():
                    valor = re.sub(r"\s*,\s*", ", ", f[rasgo].strip().lstrip("_"))
                    filas.append({
                        "doc": p.stem, "atributo": f"{tipo}.{rasgo}",
                        "texto": f"{tipo}.{rasgo}: " + con_marcas(text, s, e),
                        "valor": valor,
                    })
    return filas


# ----------------------------------------------------------------------------
# Inferencia: de etiquetas por palabra a fragmentos y a una salida anidada
# ----------------------------------------------------------------------------
def segment_bounds_inferencia(tokens, text, max_tokens=200, min_tokens=120):
    """Fragmentos para la inferencia, sin conocer las entidades: no se corta en
    cada línea en blanco (eso partiría las fichas que llevan una línea en blanco
    entre la imagen y el pie), sino solo al acercarse a max_tokens, eligiendo el
    salto de línea más «fuerte» (preferiblemente doble) a partir de min_tokens."""
    bounds, start, n = [], 0, len(tokens)
    while start < n:
        if n - start <= max_tokens:
            bounds.append((start, n)); break
        mejor, puntos = None, -1
        for i in range(start + min_tokens, min(start + max_tokens, n - 1)):
            gap = text[tokens[i][2]:tokens[i + 1][1]]
            p = gap.count("\n")
            if p > 0 and p >= puntos:
                mejor, puntos = i + 1, p
        corte = mejor or start + max_tokens
        bounds.append((start, corte)); start = corte
    return bounds


def bio_a_spans(tokens, tags):
    """tokens = [(palabra, ini, fin)], tags = ['B-PER', ...] -> [(ini, fin, 'PER')]"""
    spans, cur = [], None
    for (w, s, e), tag in zip(tokens, tags):
        if tag.startswith("B-") or (tag.startswith("I-") and (cur is None or cur[2] != tag[2:])):
            if cur:
                spans.append(tuple(cur))
            cur = [s, e, tag[2:]]
        elif tag.startswith("I-") and cur:
            cur[1] = e
        else:
            if cur:
                spans.append(tuple(cur))
            cur = None
    if cur:
        spans.append(tuple(cur))
    return spans


def montar_salida(text, spans_n1, spans_n2, atributos=None):
    """Combina niveles 1, 2 y 3 en una estructura anidada (al estilo del manual).
    atributos: dict {(ini, fin): {'person.role': 'creator', ...}}"""
    atributos = atributos or {}
    udoc = unidad_por_defecto(text, [text[s:e] for s, e, lab in spans_n2 if lab == "DIM"])

    def entidad(s, e, lab):
        d = {"tipo": lab, "texto": text[s:e].strip(), "ini": s, "fin": e}
        if lab == "DIM":
            d["valores"] = parse_dimensions(text[s:e], udoc)
        if lab == "DATE":
            d["valores"] = parse_date(text[s:e])
        if (s, e) in atributos:
            d["atributos"] = atributos[(s, e)]
        return d

    bloques = []
    for s, e, lab in spans_n1:
        hijos = [entidad(s2, e2, l2) for s2, e2, l2 in spans_n2 if s <= s2 and e2 <= e]
        b = {"tipo": {"ITEM": "catalogueItem", "EXHIB": "exhibition"}.get(lab, lab),
             "texto": text[s:e].strip()[:300], "ini": s, "fin": e, "contenido": hijos}
        if (s, e) in atributos:
            b["atributos"] = atributos[(s, e)]
        bloques.append(b)
    dentro = {(s2, e2) for s, e, _ in spans_n1 for s2, e2, _ in spans_n2 if s <= s2 and e2 <= e}
    sueltas = [entidad(s, e, l) for s, e, l in spans_n2 if (s, e) not in dentro]
    return {"bloques": bloques, "entidades_fuera_de_bloques": sueltas}


## 1. Preparación de datos y división por catálogo (80 / 10 / 10)
La división se hace por catálogo completo y es la misma para los tres niveles, de modo que los catálogos de prueba no se ven nunca durante el entrenamiento.

In [ ]:
import json, random, collections
import complexhibit_ie as ie

ie.construir_bio(CARPETA_XML, ie.NIVEL2, 'n2.jsonl')
ie.construir_bio(CARPETA_XML, ie.NIVEL1, 'n1.jsonl')
atr = ie.ejemplos_atributos(CARPETA_XML)
print(len(atr), 'ejemplos de atributos:', dict(collections.Counter(a['atributo'] for a in atr)))

docs = sorted({json.loads(l)['doc'] for l in open('n2.jsonl', encoding='utf-8')})
n = len(docs)
if os.path.exists(CARPETA_SALIDA + '/particion.json'):   # reanudación: misma partición que la vez anterior
    _p = json.load(open(CARPETA_SALIDA + '/particion.json'))
    DOCS_TEST, DOCS_VAL = set(_p['test']), set(_p['validation'])
    n_test, n_val = len(DOCS_TEST), len(DOCS_VAL)
    print('Se reutiliza la partición guardada')
else:
    random.Random(42).shuffle(docs)
    n_test = max(1, round(n * .10)); n_val = max(1, round(n * .10))
    DOCS_TEST, DOCS_VAL = set(docs[:n_test]), set(docs[n_test:n_test + n_val])
particion = lambda d: 'test' if d in DOCS_TEST else ('validation' if d in DOCS_VAL else 'train')
if not os.path.exists(CARPETA_SALIDA + '/particion.json'):
    json.dump({'test': sorted(DOCS_TEST), 'validation': sorted(DOCS_VAL)}, open(CARPETA_SALIDA + '/particion.json', 'w'), indent=1)
print(f'{n} catálogos: {n - n_test - n_val} entrenamiento · {n_val} validación · {n_test} prueba')

## 2. Funciones comunes de entrenamiento y predicción

In [ ]:
import numpy as np, torch, gc, pandas as pd
from datasets import Dataset, DatasetDict
from transformers import (AutoTokenizer, AutoModelForTokenClassification, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorForTokenClassification, DataCollatorWithPadding, set_seed)
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score
from sklearn.metrics import precision_recall_fscore_support, accuracy_score, f1_score as f1_sk

import inspect, zipfile
from google.colab import files
from transformers import TrainingArguments as _TA_original
_admitidos = set(inspect.signature(_TA_original.__init__).parameters)
def TrainingArguments(**kw):
    """Compatibilidad con versiones nuevas de transformers: ignora parámetros que ya no existen."""
    quitados = [k for k in kw if k not in _admitidos]
    if quitados:
        print('Parámetros ignorados por la versión instalada:', quitados)
    return _TA_original(**{k: v for k, v in kw.items() if k in _admitidos})

def descargar(prefijo):
    """Guarda las métricas de un nivel al terminar su entrenamiento."""
    nombre = f'/content/{prefijo}_resultados.zip'
    with zipfile.ZipFile(nombre, 'w', zipfile.ZIP_DEFLATED) as z:
        for f in glob.glob(f'{CARPETA_SALIDA}/{prefijo}*') + glob.glob(f'{CARPETA_SALIDA}/particion.json'):
            if os.path.isfile(f):
                z.write(f, os.path.basename(f))
    files.download(nombre)

MODELO_BASE = 'FacebookAI/xlm-roberta-base'   # multilingüe: catálogos en inglés y en español
set_seed(42)
tok = AutoTokenizer.from_pretrained(MODELO_BASE)
FP16 = torch.cuda.is_available()

def cargar_bio(fichero):
    filas = [json.loads(l) for l in open(fichero, encoding='utf-8')]
    tipos = sorted({t[2:] for f in filas for t in f['tags'] if t != 'O'})
    etiquetas = ['O'] + [p + t for t in tipos for p in ('B-', 'I-')]
    l2i = {e: i for i, e in enumerate(etiquetas)}
    ds = DatasetDict({s: Dataset.from_list([{'tokens': f['tokens'], 'labels': [l2i[t] for t in f['tags']]}
                                            for f in filas if particion(f['doc']) == s]) for s in ['train', 'validation', 'test']})
    return ds, etiquetas

def alinear(ej):
    enc = tok(ej['tokens'], is_split_into_words=True, truncation=True, max_length=512)
    todas = []
    for i, labs in enumerate(ej['labels']):
        prev, ids = None, []
        for w in enc.word_ids(batch_index=i):
            ids.append(-100 if w is None or w == prev else labs[w]); prev = w
        todas.append(ids)
    enc['labels'] = todas
    return enc

def a_etiquetas(pred, lab, etiquetas):
    P, L = [], []
    for pr, lr in zip(pred, lab):
        P.append([etiquetas[p] for p, l in zip(pr, lr) if l != -100])
        L.append([etiquetas[l] for p, l in zip(pr, lr) if l != -100])
    return P, L

def entrenar_ner(fichero, nombre, epocas):
    destino = f'{MODELOS}/{nombre}'
    resumen = f'{CARPETA_SALIDA}/{nombre}_metricas_resumen.json'
    if os.path.exists(resumen) and os.path.exists(destino + '/config.json'):
        print(f'{nombre}: ya entrenado y evaluado; se reutiliza el resultado guardado')
        return json.load(open(resumen))
    ds, etiquetas = cargar_bio(fichero)
    print(nombre, {s: len(ds[s]) for s in ds}, 'etiquetas:', etiquetas)
    tds = ds.map(alinear, batched=True, remove_columns=['tokens', 'labels'])
    def metricas(ev):
        P, L = a_etiquetas(np.argmax(ev[0], -1), ev[1], etiquetas)
        return {'precision': precision_score(L, P), 'recall': recall_score(L, P), 'f1': f1_score(L, P)}
    model = AutoModelForTokenClassification.from_pretrained(MODELO_BASE, num_labels=len(etiquetas),
                id2label=dict(enumerate(etiquetas)), label2id={e: i for i, e in enumerate(etiquetas)})
    args = TrainingArguments(output_dir=f'tmp_{nombre}', learning_rate=3e-5, num_train_epochs=epocas,
                             per_device_train_batch_size=8, per_device_eval_batch_size=16, weight_decay=0.01,
                             eval_strategy='epoch', save_strategy='epoch', load_best_model_at_end=True,
                             metric_for_best_model='f1', save_total_limit=1, fp16=FP16, report_to='none', seed=42)
    tr = Trainer(model=model, args=args, train_dataset=tds['train'], eval_dataset=tds['validation'],
                 data_collator=DataCollatorForTokenClassification(tok), compute_metrics=metricas)
    tr.train()
    # --- evaluación en prueba ---
    out = tr.predict(tds['test'])
    P, L = a_etiquetas(np.argmax(out.predictions, -1), out.label_ids, etiquetas)
    informe = classification_report(L, P, digits=3)
    pl = [x for s in L for x in s]; pp = [x for s in P for x in s]
    wp, wr, wf, _ = precision_recall_fscore_support(pl, pp, average='micro', labels=[e for e in etiquetas if e != 'O'], zero_division=0)
    res = {'entidad_micro': {'precision': float(precision_score(L, P)), 'recall': float(recall_score(L, P)), 'f1': float(f1_score(L, P))},
           'palabra_micro': {'precision': float(wp), 'recall': float(wr), 'f1': float(wf)}, 'fragmentos': {s: len(ds[s]) for s in ds}}
    print(f'=== {nombre}: por entidad completa ===\n{informe}')
    print(f'=== {nombre}: por palabra (sin O) === precisión {wp:.3f} · exhaustividad {wr:.3f} · F1 {wf:.3f}')
    pd.DataFrame(classification_report(L, P, output_dict=True)).T.to_csv(f'{CARPETA_SALIDA}/{nombre}_metricas_por_etiqueta.csv')
    json.dump(res, open(f'{CARPETA_SALIDA}/{nombre}_metricas_resumen.json', 'w'), indent=1)
    open(f'{CARPETA_SALIDA}/{nombre}_informe.txt', 'w').write(informe)
    tr.save_model(destino); tok.save_pretrained(destino)   # se guarda el modelo
    del tr, model; gc.collect(); torch.cuda.empty_cache()
    descargar(nombre)
    return res

def predecir_bio(modelo, texto, max_tokens=200):
    """Etiqueta un texto completo con un modelo BIO. Devuelve (tokens, etiquetas)."""
    tokens = ie.tokenize(texto)
    tags = ['O'] * len(tokens)
    id2label = modelo.config.id2label
    for a, b in ie.segment_bounds(tokens, texto, None, max_tokens):
        palabras = [ie.fix_token(t[0]) for t in tokens[a:b]]
        enc = tok(palabras, is_split_into_words=True, truncation=True, max_length=512, return_tensors='pt').to(modelo.device)
        with torch.no_grad():
            pred = modelo(**enc).logits.argmax(-1)[0].tolist()
        prev = None
        for j, w in enumerate(enc.word_ids(0)):
            if w is not None and w != prev:
                tags[a + w] = id2label[pred[j]]
            prev = w
    return tokens, tags

## 3. Nivel 2 — Entidades

In [ ]:
RES = {}
RES['nivel2'] = entrenar_ner('n2.jsonl', 'nivel2', EPOCAS_N2)

## 4. Nivel 1 — Bloques (`catalogueItem`, `exhibition`)

In [ ]:
RES['nivel1'] = entrenar_ner('n1.jsonl', 'nivel1', EPOCAS_N1)

## 5. Nivel 3 — Reglas de dimensiones y fechas
Se comparan los valores que producen las reglas con los anotados a mano (alto, ancho, fondo, peso, unidad; `when`, `from`, `to`). Las reglas no se entrenan, así que se evalúan sobre los catálogos de prueba y, como referencia, sobre todo el corpus. Los desacuerdos se guardan para revisarlos: algunos pueden ser errores de la anotación.

In [ ]:
r_test = ie.evaluar_reglas(CARPETA_XML, DOCS_TEST)
r_todo = ie.evaluar_reglas(CARPETA_XML)
RES['nivel3_reglas'] = {'prueba': {k: v for k, v in r_test.items() if k != 'errores'},
                        'corpus_completo': {k: v for k, v in r_todo.items() if k != 'errores'}}
json.dump(RES['nivel3_reglas'], open(CARPETA_SALIDA + '/nivel3_reglas_metricas.json', 'w'), indent=1)
json.dump(r_todo['errores'], open(CARPETA_SALIDA + '/nivel3_reglas_desacuerdos.json', 'w'), indent=1, ensure_ascii=False)
print(json.dumps(RES['nivel3_reglas'], indent=1))
for k, errs in r_todo['errores'].items():
    print(f'\n--- Desacuerdos en {k} (primeros 10) ---')
    for e in errs[:10]: print(e)
descargar('nivel3_reglas')

## 6. Nivel 3 — Clasificador de roles y tipos
Un único modelo recibe el atributo que se pide y el fragmento con su contexto (la entidad va marcada entre `[E]` y `[/E]`) y elige el valor. Se compara con una línea base que asigna siempre el valor más frecuente.

In [ ]:
DEST_ATR = MODELOS + '/atributos'
if os.path.exists(CARPETA_SALIDA + '/nivel3_atributos_metricas.csv') and os.path.exists(DEST_ATR + '/config.json'):
    print('Clasificador de atributos: ya entrenado; se reutiliza el resultado guardado')
    tabla_atr = pd.read_csv(CARPETA_SALIDA + '/nivel3_atributos_metricas.csv')
    RES['nivel3_atributos'] = tabla_atr.to_dict('records')
else:
    clases = sorted({f"{a['atributo']}={a['valor']}" for a in atr})
    c2i = {c: i for i, c in enumerate(clases)}
    mascara = {at: np.array([c.startswith(at + '=') for c in clases]) for at in {a['atributo'] for a in atr}}
    dsa = DatasetDict({s: Dataset.from_list([{'text': a['texto'], 'label': c2i[f"{a['atributo']}={a['valor']}"], 'atributo': a['atributo']}
                                             for a in atr if particion(a['doc']) == s]) for s in ['train', 'validation', 'test']})
    print({s: len(dsa[s]) for s in dsa}, '·', len(clases), 'valores posibles')
    tdsa = dsa.map(lambda b: tok(b['text'], truncation=True, max_length=256), batched=True, remove_columns=['text', 'atributo'])

    def pred_enmascarada(logits, atributos):
        return np.array([np.where(mascara[at], lg, -1e9).argmax() for lg, at in zip(logits, atributos)])

    def met_atr(ev):
        ats = dsa['validation']['atributo'] if len(ev[0]) == len(dsa['validation']) else dsa['test']['atributo']
        p = pred_enmascarada(ev[0], ats)
        return {'accuracy': accuracy_score(ev[1], p)}

    ma = AutoModelForSequenceClassification.from_pretrained(MODELO_BASE, num_labels=len(clases),
            id2label=dict(enumerate(clases)), label2id=c2i)
    args = TrainingArguments(output_dir='tmp_atr', learning_rate=3e-5, num_train_epochs=EPOCAS_ATR, per_device_train_batch_size=16,
                             per_device_eval_batch_size=32, weight_decay=0.01, eval_strategy='epoch',
                             save_strategy='epoch', load_best_model_at_end=True, metric_for_best_model='accuracy',
                             save_total_limit=1, fp16=FP16, report_to='none', seed=42)
    tra = Trainer(model=ma, args=args, train_dataset=tdsa['train'], eval_dataset=tdsa['validation'],
                  data_collator=DataCollatorWithPadding(tok), compute_metrics=met_atr)
    tra.train()

    out = tra.predict(tdsa['test'])
    pred = pred_enmascarada(out.predictions, dsa['test']['atributo'])
    gold = np.array(dsa['test']['label'])
    filas_t = []
    for at in sorted(mascara):
        idx = [i for i, a in enumerate(dsa['test']['atributo']) if a == at]
        if not idx: continue
        tr_vals = [l for l, a in zip(dsa['train']['label'], dsa['train']['atributo']) if a == at]
        mayoritaria = collections.Counter(tr_vals).most_common(1)[0][0] if tr_vals else -1
        filas_t.append({'atributo': at, 'n_prueba': len(idx),
                        'exactitud_modelo': float(accuracy_score(gold[idx], pred[idx])),
                        'F1_macro_modelo': float(f1_sk(gold[idx], pred[idx], average='macro', zero_division=0)),
                        'exactitud_linea_base': float(accuracy_score(gold[idx], [mayoritaria] * len(idx)))})
    tabla_atr = pd.DataFrame(filas_t)
    tabla_atr.to_csv(CARPETA_SALIDA + '/nivel3_atributos_metricas.csv', index=False)
    descargar('nivel3_atributos')
    RES['nivel3_atributos'] = filas_t
    tra.save_model(DEST_ATR); tok.save_pretrained(DEST_ATR)   # se guarda el modelo
    del tra, ma; gc.collect(); torch.cuda.empty_cache()
    tabla_atr
tabla_atr

## 7. Los tres niveles juntos: salida anidada
Se aplican los tres modelos y las reglas a un catálogo de prueba (que ningún modelo ha visto) y se guarda la estructura resultante en JSON. Si existe `CARPETA_TXT`, se procesan también los textos sin etiquetar.

In [ ]:
m1 = AutoModelForTokenClassification.from_pretrained(MODELOS + '/nivel1').to('cuda' if FP16 else 'cpu').eval()
m2 = AutoModelForTokenClassification.from_pretrained(MODELOS + '/nivel2').to(m1.device).eval()
m3 = AutoModelForSequenceClassification.from_pretrained(MODELOS + '/atributos').to(m1.device).eval()
clases_m3 = [m3.config.id2label[i] for i in range(len(m3.config.id2label))]

# qué atributos se piden para cada tipo de entidad o bloque
PEDIR = {'PER': ['person.role'], 'ORG': ['institution.type', 'institution.role'],
         'TECH': ['technique.type'], 'ITEM': ['work.type']}

def clasificar(texto, s, e, atributo):
    entrada = f'{atributo}: ' + ie.con_marcas(texto, s, e)
    enc = tok(entrada, truncation=True, max_length=256, return_tensors='pt').to(m3.device)
    with torch.no_grad():
        lg = m3(**enc).logits[0].cpu().numpy()
    ok = np.array([c.startswith(atributo + '=') for c in clases_m3])
    if not ok.any(): return None
    return clases_m3[int(np.where(ok, lg, -1e9).argmax())].split('=', 1)[1]

def procesar_documento(texto):
    tk, t1 = predecir_bio(m1, texto); sp1 = ie.bio_a_spans(tk, t1)
    tk, t2 = predecir_bio(m2, texto); sp2 = ie.bio_a_spans(tk, t2)
    atributos = {}
    for s, e, lab in sp1 + sp2:
        for at in PEDIR.get(lab, []):
            v = clasificar(texto, s, e, at)
            if v: atributos.setdefault((s, e), {})[at] = v
    return ie.montar_salida(texto, sp1, sp2, atributos)

ejemplo = sorted(DOCS_TEST)[0]
ruta = [p for p in glob.glob(CARPETA_XML + '/**/*.xml', recursive=True) if os.path.splitext(os.path.basename(p))[0] == ejemplo][0]
texto, _ = ie.load_gate(ruta)
salida = procesar_documento(texto)
json.dump(salida, open(f'{CARPETA_SALIDA}/ejemplo_salida_{ejemplo}.json', 'w'), indent=1, ensure_ascii=False)
items = [b for b in salida['bloques'] if b['tipo'] == 'catalogueItem']
print(f'{ejemplo}: {len(salida["bloques"])} bloques ({len(items)} obras) · {len(salida["entidades_fuera_de_bloques"])} entidades fuera de bloques')
print(json.dumps(items[:2], indent=1, ensure_ascii=False))

txts = glob.glob(CARPETA_TXT + '/**/*.txt', recursive=True) if os.path.isdir(CARPETA_TXT) else []
if txts:
    os.makedirs(CARPETA_SALIDA + '/salidas_txt', exist_ok=True)
    for p in txts:
        res = procesar_documento(open(p, encoding='utf-8', errors='ignore').read())
        json.dump(res, open(CARPETA_SALIDA + '/salidas_txt/' + os.path.basename(p) + '.json', 'w'), indent=1, ensure_ascii=False)
    print(len(txts), 'textos sin etiquetar procesados en', CARPETA_SALIDA + '/salidas_txt')

## 8. Resumen para el informe

In [ ]:
filas = []
for niv in ['nivel1', 'nivel2']:
    r = RES[niv]
    filas.append({'Nivel': niv, 'Medida': 'Por entidad (micro)', 'Precisión': r['entidad_micro']['precision'],
                  'Exhaustividad': r['entidad_micro']['recall'], 'F1': r['entidad_micro']['f1']})
    filas.append({'Nivel': niv, 'Medida': 'Por palabra (micro, sin O)', 'Precisión': r['palabra_micro']['precision'],
                  'Exhaustividad': r['palabra_micro']['recall'], 'F1': r['palabra_micro']['f1']})
for k, v in RES['nivel3_reglas']['prueba'].items():
    filas.append({'Nivel': 'nivel3 reglas', 'Medida': f'{k}: exactitud (n={v["n"]})', 'F1': v['exactitud']})
for r in RES['nivel3_atributos']:
    filas.append({'Nivel': 'nivel3 atributos', 'Medida': f"{r['atributo']}: exactitud (n={r['n_prueba']}; base {r['exactitud_linea_base']:.2f})",
                  'F1': r['exactitud_modelo']})
resumen = pd.DataFrame(filas).round(3)
resumen.to_csv(CARPETA_SALIDA + '/RESUMEN_tres_niveles.csv', index=False)
json.dump(RES, open(CARPETA_SALIDA + '/RESUMEN_tres_niveles.json', 'w'), indent=1, default=float)
resumen

## 9. Descargar los resultados
Se descarga un ZIP con métricas, informes, partición y salidas JSON (sin los modelos, que pesan mucho).

In [ ]:
import zipfile
with zipfile.ZipFile('/content/resultados_sin_modelos.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for raiz, _, ficheros in os.walk(CARPETA_SALIDA):
        if '/modelos' in raiz:
            continue
        for f in ficheros:
            p = os.path.join(raiz, f)
            z.write(p, os.path.relpath(p, CARPETA_SALIDA))
files.download('/content/resultados_sin_modelos.zip')